# KAT-VAD v5 — **Stage 1**: Nexar V7 grids, gates S1-G1/G2/G4/G5, R1 × **1 seed (s2024) first**

Rules: `core/docs/v5/PREREG_V5.md` **§2** (signed 2026-10-27; R7 not run, λ = 0.5 / M = 8 fixed, DVS `span`) and
**§2.4** (build notes B1–B9; B8–B9 written after the first S1-G1 read-out). R1 is trained through v5 (B9: v2 A2 is no longer borrowed).
**Fast path:** `N_SEEDS = 1` trains s2024 only; set `N_SEEDS = 3` and re-run step 6 to add s2025–2026
(s2024 is skipped as verified). Stage 2 still needs all three before any §2 read. Any gate that fails → **stop**: nothing in Stage 2 is
trained or read.

| step | what | gate | GPU? |
|---|---|---|---|
| 2 | V7 grids: `nexar_train` positives at `s1[::8]` (training rows) + `nexar_val` at **every frame** (B1) | **S1-G1** decoded length = v2's (cos vs v2 printed only, B9) | yes |
| 3 | `v5_head.npz` = V2-S `fc_norm` + Nexar A2's `c`, `m_u`, `σ_u` (D1) | — | no |
| 4 | same start: synthetic (a) + 100 real windows (b) | **S1-G2** | no |
| 5 | map tools on Stage 0's maps + official STAUC | **S1-G5** | no |
| 6 | **R1** (`v5.readout=mean`) s2024 (`N_SEEDS = 1`) trained through the v5 path | **S1-G4** J10 + config | yes |
| 7 | summary → **paste `REPORTS/stage1_summary.md` to Claude** | — | no |

**Drive layout (v5):** upload `kat-vad/` (the v5 branch) to **`Thesis-V5/kat-vad`**. Everything v5 writes goes to
`Thesis-V5/`: runs + `REPORTS/` in `Thesis-V5/outputs/v5/v5_stage1/`, new Nexar grids in
`Thesis-V5/cache/video_grid/<V2-S>/Nexar_*`. Inputs are only **read** from the old folders: `Thesis/` (data, CLIP cache,
ckpts) and `Thesis-V2/` (v2 caches, `Nexar_window`, A2 bakes, KNN, census/build reports, v2 A2 runs, Stage 0's DoTA-CAP
grids and maps). Every step skips work already on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (Nexar raw, DoTA), cache/clip, ckpts
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: v2 caches, data/Nexar_window, outputs (v2 + Stage 0)
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # code (kat-vad/) + everything v5 writes (outputs/, cache/video_grid)
os.environ['PROJECT_ROOT']       = DRIVE_V5
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V5}/outputs'

os.environ['REPO'] = f'{DRIVE_V5}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5 branch there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_ROWS'), 'this upload predates v5 Stage 1 -- re-upload the code'
prereg = (REPO / 'core/docs/v5/PREREG_V5.md').read_text(encoding='utf-8')
assert '**Status: SIGNED' in prereg.split('## §2 Stages 1–2', 1)[1], 'PREREG_V5 §2 is not signed in this upload'
assert '### §2.4' in prereg, 'upload has no §2.4 build notes'

ENC      = constants.V5_ENCODER                                   # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW                    # 'window'
DATASET  = constants.NEXAR_DATASET
V2_CACHE = Path(DRIVE_V2) / 'cache'                               # read-only
V2_OUT   = Path(DRIVE_V2) / 'outputs'                             # read-only (v2 reports, A2 runs, v5 Stage 0)
V5_CACHE = Path(DRIVE_V5) / 'cache'                               # new v5 grids
RAW      = constants.DATA_ROOT / DATASET / 'raw'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
VIDEO_S1 = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                  # v2 Nexar A2's input (x ; c u~ / sigma_u)
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
GRIDS    = V5_CACHE / constants.V5_GRID_CACHE_DIRNAME / ENC       # written here
TRAIN_GRID = GRIDS / f'{DATASET}_s{constants.V5_NEXAR_TRAIN_STRIDE}_grid{constants.V5_GRID_COARSE}'
VAL_GRID   = GRIDS / f'{DATASET}_s{constants.V5_NEXAR_VAL_STRIDE}_grid{constants.V5_GRID_COARSE}'
DOTA_GRID  = V2_CACHE / constants.V5_GRID_CACHE_DIRNAME / ENC / constants.V5_GRID_DATASET_TAG   # Stage 0's (read-only)
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{constants.V2_SPLIT_DOTA_CAP_DEV}_s3_from_s1_nexar'   # nexar_zeroshot step 2
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'                                     # v2 Nexar A2 s<seed>/
STAGE0   = V2_OUT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME                  # Stage 0 (Thesis-V2)
ROOT     = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE1_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/v5s1')
N_SEEDS  = 1                                                     # fast path: s2024 only; 3 = full §2 set
SEEDS    = list(constants.V5_RUN_SEEDS[:N_SEEDS])
for p in (REPORTS, LOCAL, GRIDS):
    p.mkdir(parents=True, exist_ok=True)
assert constants.OUTPUT_ROOT == Path(DRIVE_V5) / 'outputs', constants.OUTPUT_ROOT

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']                     # 1,749 = 33 x 53 (v2 §29)
for name, path in (('Nexar raw', RAW), ('census', CENSUS), ('Nexar_window', DATA), ('clip s1', CLIP_S1),
                   ('V2-S s1', VIDEO_S1), ('A2 bake', A2_BAKE), ('KNN', KNN), ('DoTA grids', DOTA_GRID),
                   ('DoTA s1', DOTA_S1), ('DoTA-CAP-dev A2 bake', DOTA_A2), ('v2 A2 runs', A2_RUNS),
                   ('Stage 0 maps', STAGE0 / 'maps')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}   OK')
print(f'plan: {EPOCHS} epochs = {STEPS} steps\nwrites -> grids {GRIDS}\n          runs + REPORTS {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 2, 6, 7 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF: its absl handler swallows INFO logs


def run(cmd, log=None, quiet_steps=False):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def gate(name):
    '''Read a gate's json from REPORTS and stop the notebook on FAIL (PREREG_V5 §2.2).'''
    r = json.loads((REPORTS / f'{name}.json').read_text())
    print((REPORTS / f'{name}.md').read_text())
    assert r['pass'], f'{name} FAILED -- stop here (PREREG_V5 §2.2); paste {name}.md to Claude'
    return r


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')

## 2. V7 grids + **S1-G1** (GPU)

Training rows: the sources of `Nexar_window`'s windows (positives only) at `s1[::8]`. `nexar_val`: all 225 videos at
**every frame** (§2.4 B1 — crops start on any native phase, so `s1[::3]` from phase 0 cannot serve them). S1-G1
(§2.4 B9): a video whose decoded length differs from v2's stores nothing and fails the step; the cosine against
`Nexar_s1_squash` is printed only (it differs from 1 by ≤ 6e-4 where today's decode differs from v2's).
Expected size: train ≈ 2.5 GB, val ≈ 10 GB (Drive, `Thesis-V5/cache`).

In [ ]:
from core.data.v2_splits import load_split
from core.data.windows import load_windows

census = json.loads(CENSUS.read_text())
TRAIN_IDS = sorted({w.source for w in load_windows(DATA).values()})
VAL_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_VAL))
print(f'train sources {len(TRAIN_IDS)} (positives) | nexar_val {len(VAL_IDS)}')
assert all(census['videos'][v]['label'] == 1 for v in TRAIN_IDS), 'a window source is not a positive'

def extracted(stride, ids):
    '''A PASSing S1-G1 report over exactly ``ids``: skip the step without opening ~12 GB of grids on Drive.'''
    path = REPORTS / f'g1_extract_s{stride}.json'
    if not path.exists():
        return False
    r = json.loads(path.read_text())
    return r['pass'] and r['videos'] == len(ids) and r['ok'] == len(ids)


wrote = False
for stride, ids, grid_dir, tag in ((constants.V5_NEXAR_TRAIN_STRIDE, TRAIN_IDS, TRAIN_GRID, 'train'),
                                   (constants.V5_NEXAR_VAL_STRIDE, VAL_IDS, VAL_GRID, 'val')):
    if extracted(stride, ids):
        print(f'{tag}: S1-G1 already PASS on {len(ids)} videos -- skipped')
        gate(f'g1_extract_s{stride}')
        continue
    wrote = True
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'extract', '--root', RAW, '--census', CENSUS,
         '--ids-file', write_ids(LOCAL / f'ids_{tag}.txt', ids), '--video-s1-dir', VIDEO_S1,
         '--grid-dir', grid_dir, '--stride', stride, '--report', REPORTS / f'extract_{tag}.json',
         '--batch-size', '32', '--device', 'auto', '--out-dir', REPORTS], log=LOCAL / f'extract_{tag}.log')
    print(f'{tag}: {(time.time() - t0) / 60:.0f} min')
    gate(f'g1_extract_s{stride}')
if wrote:
    flush_drive()

## 3. Motion head (D1): V2-S `fc_norm` + Nexar A2's baked `c`, `m_u`, `σ_u` → `v5_head.npz` in the train grid cache

In [ ]:
if not (TRAIN_GRID / constants.V5_HEAD_FILENAME).exists():
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'head', '--stats-dir', A2_BAKE,
         '--grid-dir', TRAIN_GRID, '--out-dir', REPORTS])
print(json.loads((REPORTS / 'head.json').read_text()))

## 4. **S1-G2** same start (CPU, VM disk)

(a) synthetic: every query readout at init is the tile mean (|Δ| ≤ 1e-6, M ∈ {1, 8}, both key types) and a DVS
splice keeps each item's own ΔG; (b) 100 real Nexar windows: every row's motion term at init equals R1's on the
same grid (|Δ| ≤ 1e-3).

In [ ]:
def staged():
    '''Training inputs on VM disk: copied once per VM, and only when a step below actually runs.'''
    return (stage(DATA, LOCAL / 'stage' / f'data_{C}'), stage(A2_BAKE, LOCAL / 'stage' / 'A2_s8'),
            stage(TRAIN_GRID, LOCAL / 'stage' / 'grid_s8'), stage(KNN, LOCAL / 'stage' / f'knn_{C}.npz'))


if not (REPORTS / 'g2_init.json').exists():
    L_DATA, L_A2, L_GRID, L_KNN = staged()
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'init-check', '--data-dir', L_DATA, '--clip-dir', L_A2,
         '--grid-dir', L_GRID, '--out-dir', REPORTS])
gate('g2_init')


## 5. **S1-G5** map tools (CPU)

(a) the S2-8 read fed Stage 0's stored V7 change maps reproduces `RESULTS_V5_STAGE0.md` §3's V7 D3 row (|Δ| ≤ 1e-4);
(b) random maps do not point, planted in-box maps do; (c) the STAUC port matches DoTA's official `stauc.py`
(`data/DoTA/raw_annotations/stauc.py` on Drive) on a fixed synthetic input (|Δ| ≤ 1e-6).

In [ ]:
if not (REPORTS / 'g5_maps.json').exists():
    RAW_ANN = DOTA / 'raw_annotations'                 # on Drive: DoTA's official stauc.py + annotations/
    OFFICIAL = RAW_ANN / 'stauc.py'
    ANN = RAW_ANN / 'annotations'
    DOTA_LOCAL = LOCAL / 'dota'
    DOTA_LOCAL.mkdir(parents=True, exist_ok=True)
    assert OFFICIAL.is_file(), f'missing DoTA official {OFFICIAL} (S1-G5(c) compares against it)'
    if not ANN.is_dir():
        ANN = DOTA_LOCAL / 'annotations'
        if not ANN.exists():
            subprocess.run(f'cd {DOTA_LOCAL} && curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip',
                           shell=True, check=True)
    META = DOTA / 'metadata_val.json'
    SPLIT = DOTA_LOCAL / 'val_ids.txt'
    SPLIT.write_text(''.join(f'{v}\n' for v in json.loads(META.read_text())))
    print('annotations:', len(list(ANN.glob('*.json'))), '| official stauc.py', OFFICIAL.stat().st_size, 'bytes')
    run([sys.executable, '-m', 'core.tools.v5_maps', 'g5', '--reports', *sorted(STAGE0.glob('extract_*.json')),
         '--maps-dir', STAGE0 / 'maps', '--stage0-readout', STAGE0 / 'REPORTS' / 'stage0_readout.json',
         '--official', OFFICIAL, '--annotations-dir', ANN, '--metadata', META, '--split-file', SPLIT,
         '--out-dir', REPORTS])
gate('g5_maps')

## 6. **R1** × `N_SEEDS` seeds through the v5 path, then **S1-G4** (GPU)

v2 Nexar A2's exact command (`window`, A2 bake, KNN, kernel 3, top-k 5, `span`, not ego, AMP) plus
`v5.readout=mean` and `--grid-dir`. Local training, then an atomic publish to Drive (lesson (bi)).

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'


def train_cmd(row, seed, out_dir):
    l_data, l_a2, l_grid, l_knn = staged()
    v5 = [x for k, v in constants.V5_ROWS[row].items() for x in ('--set', f'v5.{k}={v}')]
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={seed}',
            '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
            '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
            '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
            '--set', 'data.is_egocentric=false',
            '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}', *v5,
            '--data-dir', l_data, '--clip-dir', l_a2, '--knn-cache', l_knn, '--grid-dir', l_grid,
            '--output-dir', out_dir]


VERIFIED = 'verified.json'


def mark_verified(drive_dir):
    '''Record a J10-verified Drive run once, so later sessions skip it without loading the 1.4 GB checkpoint.'''
    rec = {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}
    (LOCAL / VERIFIED).write_text(json.dumps(rec))
    publish(LOCAL / VERIFIED, drive_dir / VERIFIED)


def is_verified(drive_dir):
    '''The run's own verify record still matches its checkpoint (a stat, not a download).'''
    rec_path, ckpt = drive_dir / VERIFIED, drive_dir / 'checkpoint_last.pt'
    if not (rec_path.exists() and ckpt.exists()):
        return False
    rec = json.loads(rec_path.read_text())
    return rec['step'] == STEPS and rec['ckpt_bytes'] == ckpt.stat().st_size


def train_row(row, seed):
    drive_dir, local = ROOT / row / f's{seed}', LOCAL / 'runs' / row / f's{seed}'
    if is_verified(drive_dir):
        print(f'{row} s{seed}: verified on Drive ({VERIFIED}) -- not retrained')
        return drive_dir
    last = metrics_last(drive_dir / 'metrics.jsonl')
    if last and ckpt_step(drive_dir / 'checkpoint_last.pt') == last['global_step'] == STEPS:
        mark_verified(drive_dir)       # runs trained before verified.json existed: one full read, then never again
        print(f'{row} s{seed}: verified on Drive -- not retrained')
        return drive_dir
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(row, seed, local) + resume, log=LOCAL / f'train_{row}_s{seed}.log', quiet_steps=True)
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row} s{seed}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{row} s{seed}: Drive copy is not step {STEPS}'
    mark_verified(drive_dir)
    print(f'{row} s{seed}: {(time.time() - t0) / 60:.0f} min, last mil {last.get("mil")}')
    return drive_dir


def provenance_done():
    '''A PASSing S1-G4 over exactly these runs (provenance re-loads every checkpoint from Drive).'''
    path = REPORTS / 'g4_provenance.json'
    return path.exists() and (r := json.loads(path.read_text()))['pass'] and \
        set(r['runs']) == {f'R1/s{s}' for s in SEEDS}


for seed in SEEDS:
    train_row('R1', seed)
if not provenance_done():
    # train_row already re-read every Drive checkpoint (J10); read this session's local copies instead of
    # pulling 1.4 GB per seed from Drive a second time. Seeds skipped this session only exist on Drive.
    local_runs = LOCAL / 'runs'
    prov_root = local_runs if all((local_runs / 'R1' / f's{s}' / 'checkpoint_last.pt').exists() for s in SEEDS) else ROOT
    run([sys.executable, '-m', 'core.tools.v5_stage1', 'provenance', '--root', prov_root, '--a2-root', A2_RUNS,
         '--rows', 'R1', '--seeds', *SEEDS, '--steps', STEPS, '--out-dir', REPORTS])
gate('g4_provenance')

## 7. Summary → paste `REPORTS/stage1_summary.md` to Claude

In [ ]:
names = ['g1_extract_s8', 'g1_extract_s1', 'g2_init', 'g5_maps', 'g4_provenance']
lines = ['# v5 Stage 1 summary (PREREG_V5 §2.2)', '']
for n in names:
    r = json.loads((REPORTS / f'{n}.json').read_text())
    lines.append(f"- **{n}**: {'PASS' if r['pass'] else 'FAIL'}" + (f" -- {r['verdict']}" if 'verdict' in r else ''))
lines += [''] + [(REPORTS / f'{n}.md').read_text() for n in names]
(REPORTS / 'stage1_summary.md').write_text('\n'.join(lines))
print('\n'.join(lines))
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- safe to close the runtime')